In [516]:
import random
from datetime import datetime, timedelta
import pandas as pd

In [517]:
# Set random seed for reproducibility
random.seed(42)

In [518]:
# Configuration parameters
NUM_PATIENTS = 200
start_date = datetime(2026, 6, 1, 8, 0, 0)

In [519]:
# Reference data pools
genders = ["Male", "Female", "Other"]
doctors = ["Dr. Robert", "Dr. House", "Dr. Smith", "Dr. Adams", "Dr. Chen"]
triage_levels = ["ESI Level 2", "ESI Level 3", "ESI Level 4", "ESI Level 5"]

In [520]:
# Locations mapped to activities
activity_locations = {
    "Check-in": "Reception Desk",
    "Triage": "Triage Room 1",
    "Doctor Consultation": "Consultation Bay B",
    "X-Ray": "Radiology Lab 2",
    "Discharge": "Exit Lobby",
}

In [521]:
resources = {
    "Check-in": "Front Desk Staff",
    "Triage": "Nurse Sarah",
    "Doctor Consultation": "Attending Physician",
    "X-Ray": "Tech Mike",
    "Discharge": "Discharge Coordinator",
}

In [522]:
# Reset event rows
event_rows = []

# Consolidated loop to generate all 200 sequential patient records (1000 to 1199)
for i in range(NUM_PATIENTS):
    case_id = 1000 + i
    patient_age = random.randint(18, 85)
    patient_gender = random.choice(genders)
    attending_doctor = random.choice(doctors)
    acuity_level = random.choice(triage_levels)

    # 40% chance that a patient is missing paperwork, triggering the rework loop after X-Ray
    paperwork_missing_flag = random.random() < 0.40

    # Stagger patient arrival times
    current_time = start_date + timedelta(
        minutes=random.randint(0, 1440 * 3)
    )  # Spread over 3 days

    # 1. Check-in
    event_rows.append({
        "case_id": case_id,
        "activity": "Check-in",
        "timestamp": current_time,
        "resource": resources["Check-in"],
        "location": activity_locations["Check-in"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 2. Triage
    current_time += timedelta(minutes=random.randint(5, 20))
    event_rows.append({
        "case_id": case_id,
        "activity": "Triage",
        "timestamp": current_time,
        "resource": resources["Triage"],
        "location": activity_locations["Triage"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 3. Doctor Consultation
    current_time += timedelta(minutes=random.randint(15, 60))
    event_rows.append({
        "case_id": case_id,
        "activity": "Doctor Consultation",
        "timestamp": current_time,
        "resource": attending_doctor,
        "location": activity_locations["Doctor Consultation"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 4. X-Ray
    current_time += timedelta(minutes=random.randint(10, 30))
    event_rows.append({
        "case_id": case_id,
        "activity": "X-Ray",
        "timestamp": current_time,
        "resource": resources["X-Ray"],
        "location": activity_locations["X-Ray"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # --- INJECTED BOTTLENECK / REWORK LOOP ---
    if paperwork_missing_flag:
        # Sent back to Triage due to missing form
        current_time += timedelta(minutes=random.randint(15, 30))
        event_rows.append({
            "case_id": case_id,
            "activity": "Triage",  # Rework step
            "timestamp": current_time,
            "resource": resources["Triage"],
            "location": activity_locations["Triage"],
            "acuity_level": acuity_level,
            "patient_age": patient_age,
            "patient_gender": patient_gender,
            "attending_doctor": attending_doctor,
            "paperwork_missing_flag": True,
        })

        # Quick re-consultation after form fix
        current_time += timedelta(minutes=random.randint(10, 25))
        event_rows.append({
            "case_id": case_id,
            "activity": "Doctor Consultation",
            "timestamp": current_time,
            "resource": attending_doctor,
            "location": activity_locations["Doctor Consultation"],
            "acuity_level": acuity_level,
            "patient_age": patient_age,
            "patient_gender": patient_gender,
            "attending_doctor": attending_doctor,
            "paperwork_missing_flag": False,
        })

    # 5. Discharge
    current_time += timedelta(minutes=random.randint(15, 40))
    event_rows.append({
        "case_id": case_id,
        "activity": "Discharge",
        "timestamp": current_time,
        "resource": resources["Discharge"],
        "location": activity_locations["Discharge"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

In [523]:
print(df_events['case_id'])

0       1000
1       1000
2       1000
3       1000
4       1000
        ... 
1137    1199
1138    1199
1139    1199
1140    1199
1141    1199
Name: case_id, Length: 1142, dtype: int64


In [524]:
# Convert to DataFrame
df_events = pd.DataFrame(event_rows)# Sort chronologically per case
df_events = df_events.sort_values(by=["case_id", "timestamp"]).reset_index(drop=True)


In [525]:
# Sort chronologically per case
df_events = df_events.sort_values(by=["case_id", "timestamp"]).reset_index(
    drop=True
)

In [526]:
# Preview the first 10 rows
print(df_events.head(10))

   case_id             activity           timestamp               resource  \
0     1000             Check-in 2026-06-01 21:59:00       Front Desk Staff   
1     1000               Triage 2026-06-01 22:06:00            Nurse Sarah   
2     1000  Doctor Consultation 2026-06-01 22:58:00              Dr. Smith   
3     1000                X-Ray 2026-06-01 23:21:00              Tech Mike   
4     1000               Triage 2026-06-01 23:37:00            Nurse Sarah   
5     1000  Doctor Consultation 2026-06-01 23:47:00              Dr. Smith   
6     1000            Discharge 2026-06-02 00:04:00  Discharge Coordinator   
7     1001             Check-in 2026-06-03 17:16:00       Front Desk Staff   
8     1001               Triage 2026-06-03 17:28:00            Nurse Sarah   
9     1001  Doctor Consultation 2026-06-03 18:11:00               Dr. Chen   

             location acuity_level  patient_age patient_gender  \
0      Reception Desk  ESI Level 3           32           Male   
1       T

In [527]:
# Preview the last 10 rows
print(df_events.tail(10))

      case_id             activity           timestamp               resource  \
1132     1198             Check-in 2026-06-02 05:33:00       Front Desk Staff   
1133     1198               Triage 2026-06-02 05:51:00            Nurse Sarah   
1134     1198  Doctor Consultation 2026-06-02 06:47:00              Dr. House   
1135     1198                X-Ray 2026-06-02 07:12:00              Tech Mike   
1136     1198            Discharge 2026-06-02 07:42:00  Discharge Coordinator   
1137     1199             Check-in 2026-06-01 20:12:00       Front Desk Staff   
1138     1199               Triage 2026-06-01 20:24:00            Nurse Sarah   
1139     1199  Doctor Consultation 2026-06-01 21:13:00              Dr. Smith   
1140     1199                X-Ray 2026-06-01 21:24:00              Tech Mike   
1141     1199            Discharge 2026-06-01 21:44:00  Discharge Coordinator   

                location acuity_level  patient_age patient_gender  \
1132      Reception Desk  ESI Level 2  

In [528]:
# Preview the output
display(df_events.head(15))

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
5,1000,Doctor Consultation,2026-06-01 23:47:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,False
6,1000,Discharge,2026-06-02 00:04:00,Discharge Coordinator,Exit Lobby,ESI Level 3,32,Male,Dr. Smith,True
7,1001,Check-in,2026-06-03 17:16:00,Front Desk Staff,Reception Desk,ESI Level 2,45,Male,Dr. Chen,False
8,1001,Triage,2026-06-03 17:28:00,Nurse Sarah,Triage Room 1,ESI Level 2,45,Male,Dr. Chen,False
9,1001,Doctor Consultation,2026-06-03 18:11:00,Dr. Chen,Consultation Bay B,ESI Level 2,45,Male,Dr. Chen,False


In [529]:
# Save to CSV for Task 1 BigQuery ingestion
df_events.to_csv("careflow_synthetic_ehr_logs.csv", index=False)
print(
    "\nDataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'."
)


Dataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'.
